## 1. Clonar llama.cpp
Clonamos el repositorio oficial `llama.cpp` de GitHub, que contiene las herramientas para convertir, cuantizar y ejecutar LLMs en hardware local. Luego nos movemos al directorio `llama.cpp`.

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 130659, done.
remote: Counting objects: 100% (1313/1313), done.
remote: Compressing objects: 100% (308/308), done.
remote: Total 130659 (delta 1089), reused 1084 (delta 1005), pack-reused 129346 (from 2)
Receiving objects: 100% (130659/130659), 437.98 MiB | 17.99 MiB/s, done.
Resolving deltas: 100% (92885/92885), done.
/content/llama.cpp


## 2. Compilar las herramientas
Configuramos el proyecto con `cmake` y lo compilamos con `cmake --build`. Esto genera los ejecutables en `build/bin/`, entre ellos `llama-quantize`, que usaremos para cuantizar.

In [ ]:
!cmake -B build
!cmake --build build --config Release -j 2

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.5.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

## 3. Descargar el modelo base de Hugging Face
Con la función `snapshot_download` de la librería `huggingface_hub` descargamos el modelo **Qwen/Qwen3.5-0.8B** en formato SafeTensors. Elegimos este modelo por ser liviano (0.8B parámetros), lo que permite trabajar cómodamente en Colab. Los archivos se guardan en `/content/qwen`.

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="Qwen/Qwen3.5-0.8B",
    local_dir="/content/qwen"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

## 4. Convertir el modelo a GGUF (sin cuantizar)
Ejecutamos el script `convert_hf_to_gguf.py` de `llama.cpp`, que convierte los pesos de SafeTensors al formato **GGUF**, el formato que usan `llama.cpp` y Ollama. Exportamos en **bf16** (16 bits, sin cuantizar), que conserva la misma precisión que el modelo original. El resultado es `/content/qwen/model.gguf`, de aproximadamente 1.5 GB.

In [ ]:
!/content/llama.cpp/convert_hf_to_gguf.py /content/qwen --outfile /content/qwen/model.gguf --outtype bf16

INFO:hf-to-gguf:Loading model: qwen
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:hf-to-gguf:Model architecture: Qwen3_5ForConditionalGeneration
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors-00001-of-00001.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                    torch.bfloat16 --> BF16, shape = {1024, 248320}
INFO:hf-to-gguf:blk.0.attn_norm.weight,               torch.bfloat16 --> F32, shape = {1024}
INFO:hf-to-gguf:blk.0.ssm_a,                          torch.float32 --> F32, shape = {16}
INFO:hf-to-gguf:blk.0.ssm_conv1d.weight,              torch.bfloat16 --> F32, shape = {4, 6144}
INFO:hf-to-gguf:blk.0.ssm_dt.bias,                    torch.bfloat16 --> F32, shape = {16}
INFO:hf-to-gguf:blk.0.ssm_alpha.weight,               torch.bfloat16 --> BF16, shape = {1024, 16

## 5. Cuantizar a 4 bits (Q4_K_M)
Usamos `llama-quantize` para reducir la precisión de los pesos a 4 bits con la variante **Q4_K_M**. Esta variante cuantiza la mayoría de los tensores en `q4_K` y deja algunos más sensibles en `q6_K` para no perder tanta calidad. El modelo pasa de ~1475 MiB (16 bits por peso) a ~506 MiB (~5.5 bits por peso en promedio), casi 3 veces más chico. Se guarda como `qwen3.5-0.8b_q4_k_m.gguf`.

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize /content/qwen/model.gguf /content/qwen/qwen3.5-0.8b_q4_k_m.gguf Q4_K_M

version: 0.5.0-dev (build 11237, commit 14ebbd5f2)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/qwen/model.gguf' to '/content/qwen/qwen3.5-0.8b_q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 43 key-value pairs and 335 tensors from /content/qwen/model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen35
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen
llama_model_loader: - kv   3:                         general.size_label str              = 773M
llama_model_loader: - kv   4:                            general.license str              = apache-2.0
llama_model_loader: - kv   5:                       general.license.link str

## 6. Verificar los archivos generados
Listamos el directorio para comparar el tamaño del modelo GGUF en bf16 con el del modelo cuantizado.

In [ ]:
!ls -lh /content/qwen/*.gguf

## 7. Probar el modelo en Ollama
Instalamos **Ollama** dentro de Colab. El script de instalación necesita `zstd` para descomprimir, así que lo instalamos primero.

In [ ]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

Ollama funciona como un servidor. Como en Colab no podemos dejar una terminal abierta, lo iniciamos en segundo plano con `subprocess` y esperamos unos segundos a que arranque.

In [ ]:
import subprocess, time

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(5)
!ollama --version

Para importar un GGUF propio, Ollama usa un **Modelfile**: un archivo de texto que indica de dónde sale el modelo (`FROM`). Lo creamos apuntando al modelo cuantizado.

In [ ]:
%%writefile /content/qwen/Modelfile
FROM /content/qwen/qwen3.5-0.8b_q4_k_m.gguf

Con `ollama create` registramos el modelo en Ollama con el nombre `qwen35-q4`, y con `ollama list` comprobamos que aparece junto con su tamaño.

In [ ]:
!ollama create qwen35-q4 -f /content/qwen/Modelfile
!ollama list

Finalmente, interactuamos con el modelo cuantizado pasándole un prompt a `ollama run`. Al pasar el prompt como argumento, Ollama responde y termina, en lugar de abrir un chat interactivo que en Colab quedaría bloqueado.

In [ ]:
!ollama run qwen35-q4 "Hola! Explicame en dos oraciones qué es la cuantización de un modelo de lenguaje."

## Conclusión
Descargamos un modelo base de Hugging Face, lo convertimos a GGUF en bf16, lo cuantizamos a 4 bits con Q4_K_M (reduciendo su tamaño casi 3 veces) y lo ejecutamos localmente con Ollama.